# 🚗 Análise Estratégica do Mercado Automotivo: Precificação e Oportunidades de Negócio

**Autor:** Gabriel Henrique  
**Área:** Business Analytics & Market Intelligence  
**Objetivo:** Diagnóstico de mercado e análise de comportamento de preços, liquidez de marcas, depreciação e eficiência comercial frente à tabela de referência (*MMR*) no setor de veículos seminovos e usados.

---

## 📌 1. Perguntas Estratégicas de Negócio
1. **Eficiência de Precificação:** As vendas estão alinhadas com o valor de referência do mercado (*MMR*) ou há queima de margem?
2. **Impacto da Conservação:** Qual o ganho financeiro real de veículos com alto padrão de conservação física?
3. **Curva de Depreciação:** Como a idade e a quilometragem corroem o valor residual dos ativos?
4. **Liquidez por Fabricante:** Quais marcas concentram maior giro de estoque?

### 📦 1. Importação das Bibliotecas e Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configurações estéticas para relatórios executivos
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado e bibliotecas carregadas com sucesso!")

### 📂 2. Carregamento dos Dados Brutos
Importação da base transacional contendo o histórico de vendas de veículos.

In [ ]:
data_path = 'data/car_prices.csv'
df = pd.read_csv(data_path, on_bad_lines='skip')

print(f"📊 Total de registros brutos: {df.shape[0]:,}")
print(f"📋 Total de variáveis coletadas: {df.shape[1]}")
df.head(3)

### 🧹 3. Pipeline de Limpeza e Engenharia de Atributos (Feature Engineering)
- Padronização de strings (marcas e tipos de carroceria).
- Tratamento de registros inconsistentes e valores nulos.
- Construção de métricas de performance comercial: `price_diff` (Desvio vs Tabela) e `margin_pct` (Variação Percentual).

In [ ]:
df_clean = df.dropna(subset=['make', 'model', 'sellingprice', 'mmr', 'year', 'odometer', 'condition']).copy()

df_clean['make'] = df_clean['make'].str.strip().str.title()
df_clean['body'] = df_clean['body'].str.strip().str.title()

df_clean['price_diff'] = df_clean['sellingprice'] - df_clean['mmr']
df_clean['margin_pct'] = (df_clean['price_diff'] / df_clean['mmr']) * 100
df_clean['vehicle_age'] = 2015 - df_clean['year']

print(f"✅ Registros válidos e tratados: {df_clean.shape[0]:,} transações.")

### 📊 4. Visão Executiva: Indicadores-Chave de Desempenho (KPIs)

In [ ]:
total_vendas_usd = df_clean['sellingprice'].sum()
ticket_medio = df_clean['sellingprice'].mean()
km_media = df_clean['odometer'].mean()
condicao_media = df_clean['condition'].mean()
margem_media = df_clean['margin_pct'].mean()

print("="*50)
print("📈 DASHBOARD EXECUTIVO - RESUMO GERAL")
print("="*50)
print(f"💰 Volume Total Transacionado: ${total_vendas_usd:,.2f}")
print(f"🏷️  Ticket Médio por Veículo:  ${ticket_medio:,.2f}")
print(f"🛣️  Quilometragem Média:       {km_media:,.0f} milhas")
print(f"⭐ Nota Média de Conservação:  {condicao_media:.1f} / 50")
print(f"📊 Variação Média vs Mercado:  {margem_media:+.2f}%")
print("="*50)

### 🏆 5. Concentração de Mercado: Top 10 Fabricantes Mais Negociados

In [ ]:
top_makes = df_clean['make'].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_makes.values, y=top_makes.index, palette='Blues_r')
plt.title('Top 10 Fabricantes por Volume de Vendas', fontsize=14, fontweight='bold')
plt.xlabel('Volume de Veículos Negociados')
plt.ylabel('Fabricante')
plt.show()

### 📉 6. Análise de Depreciação Temporal: Preço Mediano vs Ano de Fabricação

In [ ]:
preco_por_ano = df_clean.groupby('year')['sellingprice'].median().reset_index()

plt.figure(figsize=(11, 5))
sns.lineplot(data=preco_por_ano, x='year', y='sellingprice', marker='o', color='#1f77b4', linewidth=2.5)
plt.title('Curva de Depreciação por Ano do Veículo', fontsize=14, fontweight='bold')
plt.xlabel('Ano de Fabricação')
plt.ylabel('Preço Mediano de Venda ($)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 🎯 7. Avaliação de Sensibilidade: Estado de Conservação vs Preço de Venda

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean[df_clean['year'] >= 2010], x=pd.cut(df_clean['condition'], bins=5), y='sellingprice', palette='crest')
plt.title('Dispersão de Preços por Faixa de Conservação Física (Veículos >= 2010)', fontsize=14, fontweight='bold')
plt.xlabel('Faixa de Nota de Conservação (Escala 1 a 50)')
plt.ylabel('Preço de Venda ($)')
plt.show()

### 📌 8. Conclusões Estratégicas & Recomendações de Negócio

1. **Retorno sobre Preparação Estética:** Veículos no quintil superior de conservação (notas entre 39.4 e 49.0) apresentam uma valorização mediana superior a 35% frente aos veículos de conservação regular. Recomenda-se criar um processo padrão de recondicionamento e higienização para maximizar a margem antes da disponibilização no pátio.
2. **Aderência à Tabela de Mercado:** A operação apresenta desvio médio controlado de -0.73% em relação ao MMR, evidenciando competitividade de preços sem comprometer severamente a lucratividade.
3. **Foco em Liquidez Operacional:** Marcas líderes de volume (Ford, Chevrolet e Nissan) concentram mais de 35% do estoque, devendo ter giro monitorado por SLA para evitar depreciação por tempo excessivo de pátio.
4. **Direcionamento Estratégico:** Para automatizar e padronizar o processo de entrada de estoque, implementou-se o modelo preditivo no `Notebook 02`.